# TP 5 : Spark GraphX et GraphFrames

## Installation

In [1]:
%pip install -q pyspark==3.5.1 graphframes

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\pyspark\\jars\\HikariCP-2.5.1.jar'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Démarrage de la session Spark

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TP5") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12") \
    .config("spark.jars.repositories", "https://repos.spark-packages.org") \
    .getOrCreate()

## Question 1 : Créer un DataFrame pour chaque fichier

In [2]:
bikeStations = spark.read.option("header", "true").csv("station_data.csv")
tripData = spark.read.option("header", "true").csv("trip_data.csv")

bikeStations.show(5, False)
tripData.show(5, False)

+----------+---------------------------------+---------+-----------+---------+--------+------------+
|station_id|name                             |lat      |long       |dockcount|landmark|installation|
+----------+---------------------------------+---------+-----------+---------+--------+------------+
|2         |San Jose Diridon Caltrain Station|37.329732|-121.901782|27       |San Jose|8/6/2013    |
|3         |San Jose Civic Center            |37.330698|-121.888979|15       |San Jose|8/5/2013    |
|4         |Santa Clara at Almaden           |37.333988|-121.894902|11       |San Jose|8/6/2013    |
|5         |Adobe on Almaden                 |37.331415|-121.8932  |19       |San Jose|8/5/2013    |
|6         |San Pedro Square                 |37.336721|-121.894074|15       |San Jose|8/7/2013    |
+----------+---------------------------------+---------+-----------+---------+--------+------------+
only showing top 5 rows

+-------+--------+---------------+--------------------------------

## Question 2 : Renommer la colonne name en id (table des sommets)

In [3]:
stationVertices = bikeStations.withColumnRenamed("name", "id").distinct()
stationVertices.show(5, False)

+----------+------------------------+---------+-----------+---------+-------------+------------+
|station_id|id                      |lat      |long       |dockcount|landmark     |installation|
+----------+------------------------+---------+-----------+---------+-------------+------------+
|6         |San Pedro Square        |37.336721|-121.894074|15       |San Jose     |8/7/2013    |
|8         |San Salvador at 1st     |37.330165|-121.885831|15       |San Jose     |8/5/2013    |
|14        |Arena Green / SAP Center|37.332692|-121.900084|19       |San Jose     |8/5/2013    |
|25        |Stanford in Redwood City|37.48537 |-122.203288|15       |Redwood City |8/12/2013   |
|30        |Evelyn Park and Ride    |37.390277|-122.066553|15       |Mountain View|8/16/2013   |
+----------+------------------------+---------+-----------+---------+-------------+------------+
only showing top 5 rows



## Question 3 : Renommer Start Station en src et End Station en dst (table des arcs)

In [4]:
tripEdges = tripData \
    .withColumnRenamed("Start Station", "src") \
    .withColumnRenamed("End Station", "dst")
tripEdges.show(5, False)

+-------+--------+---------------+------------------------------------+--------------+---------------+----------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                 |Start Terminal|End Date       |dst                                     |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+------------------------------------+--------------+---------------+----------------------------------------+------------+------+---------------+--------+
|913460 |765     |8/31/2015 23:26|Harry Bridges Plaza (Ferry Building)|50            |8/31/2015 23:39|San Francisco Caltrain (Townsend at 4th)|70          |288   |Subscriber     |2139    |
|913459 |1036    |8/31/2015 23:11|San Antonio Shopping Center         |31            |8/31/2015 23:28|Mountain View City Hall                 |27          |35    |Subscriber     |95032   |
|913455 |307     |8/31/2015 23:13|Post at Kearny       

## Question 4 : Créer le GraphFrame

In [5]:
from graphframes import GraphFrame

stationGraph = GraphFrame(stationVertices, tripEdges)

print("Total Number of Stations: " + str(stationGraph.vertices.count()))
print("Total Number of Trips in Graph: " + str(stationGraph.edges.count()))
print("Total Number of Trips in Original Data: " + str(tripData.count()))

Total Number of Stations: 70
Total Number of Trips in Graph: 99
Total Number of Trips in Original Data: 99


## Question 5 : Nombre de voyages entre chaque source et destination (ordre décroissant)

In [6]:
from pyspark.sql.functions import desc, col

stationGraph.edges \
    .groupBy("src", "dst") \
    .count() \
    .orderBy(desc("count")) \
    .show(10, False)

+---------------------------------------+----------------------------------------+-----+
|src                                    |dst                                     |count|
+---------------------------------------+----------------------------------------+-----+
|San Francisco Caltrain 2 (330 Townsend)|Townsend at 7th                         |4    |
|5th at Howard                          |San Francisco Caltrain 2 (330 Townsend) |3    |
|San Francisco Caltrain 2 (330 Townsend)|Powell Street BART                      |2    |
|Steuart at Market                      |San Francisco Caltrain (Townsend at 4th)|2    |
|Market at Sansome                      |Broadway St at Battery St               |2    |
|Davis at Jackson                       |Embarcadero at Sansome                  |2    |
|San Francisco Caltrain 2 (330 Townsend)|2nd at South Park                       |2    |
|Embarcadero at Folsom                  |Embarcadero at Sansome                  |2    |
|Market at 4th       

## Question 6 : Nombre de voyages qui commencent ou se terminent à 'Townsend at 7th' (ordre décroissant)

In [7]:
stationGraph.edges \
    .where("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'") \
    .groupBy("src", "dst") \
    .count() \
    .orderBy(desc("count")) \
    .show(10, False)

+---------------------------------------+------------------------------------+-----+
|src                                    |dst                                 |count|
+---------------------------------------+------------------------------------+-----+
|San Francisco Caltrain 2 (330 Townsend)|Townsend at 7th                     |4    |
|Spear at Folsom                        |Townsend at 7th                     |1    |
|5th at Howard                          |Townsend at 7th                     |1    |
|Howard at 2nd                          |Townsend at 7th                     |1    |
|Townsend at 7th                        |Harry Bridges Plaza (Ferry Building)|1    |
|Townsend at 7th                        |Spear at Folsom                     |1    |
+---------------------------------------+------------------------------------+-----+



## Question 7 : Sommets qui n'ont jamais été destination d'un voyage commençant à 'Spear at Folsom'

In [8]:
destinations = stationGraph.edges \
    .where("src = 'Spear at Folsom'") \
    .select(col("dst").alias("dest_id")) \
    .distinct()

stationGraph.vertices \
    .join(destinations, stationGraph.vertices["id"] == destinations["dest_id"], "left_anti") \
    .select("id") \
    .show(100, False)

+---------------------------------------------+
|id                                           |
+---------------------------------------------+
|San Pedro Square                             |
|San Salvador at 1st                          |
|Arena Green / SAP Center                     |
|Stanford in Redwood City                     |
|Evelyn Park and Ride                         |
|San Antonio Shopping Center                  |
|Castro Street and El Camino Real             |
|California Ave Caltrain Station              |
|Park at Olive                                |
|Spear at Folsom                              |
|Beale at Market                              |
|2nd at South Park                            |
|South Van Ness at Market                     |
|Steuart at Market                            |
|Mechanics Plaza (Market at Battery)          |
|Broadway St at Battery St                    |
|Adobe on Almaden                             |
|Franklin at Maple                      

## Question 8 : Station avec le nombre maximum de voyages entrants

In [9]:
stationGraph.inDegrees \
    .orderBy(desc("inDegree")) \
    .limit(1) \
    .show(truncate=False)

+---------------------------------------+--------+
|id                                     |inDegree|
+---------------------------------------+--------+
|San Francisco Caltrain 2 (330 Townsend)|9       |
+---------------------------------------+--------+



## Question 9 : Voyage de plus grande durée

In [10]:
stationGraph.edges \
    .withColumn("Duration", col("Duration").cast("long")) \
    .orderBy(desc("Duration")) \
    .limit(1) \
    .show(truncate=False, vertical=True)

-RECORD 0-----------------------------------------------
 Trip ID         | 913386                               
 Duration        | 1808                                 
 Start Date      | 8/31/2015 20:23                      
 src             | Embarcadero at Bryant                
 Start Terminal  | 54                                   
 End Date        | 8/31/2015 20:53                      
 dst             | Harry Bridges Plaza (Ferry Building) 
 End Terminal    | 50                                   
 Bike #          | 524                                  
 Subscriber Type | Subscriber                           
 Zip Code        | 94105                                



## Question 10 : Sous-graphe des voyages qui commencent ou se terminent à 'Townsend at 7th'

In [11]:
townAnd7thEdges = stationGraph.edges \
    .where("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'")

subgraph = GraphFrame(stationGraph.vertices, townAnd7thEdges)

print("Sommets : " + str(subgraph.vertices.count()))
print("Arcs : " + str(subgraph.edges.count()))
subgraph.edges.show(5, False)

Sommets : 70
Arcs : 9
+-------+--------+---------------+---------------------------------------+--------------+---------------+------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                    |Start Terminal|End Date       |dst                                 |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+---------------------------------------+--------------+---------------+------------------------------------+------------+------+---------------+--------+
|913434 |283     |8/31/2015 21:19|San Francisco Caltrain 2 (330 Townsend)|69            |8/31/2015 21:24|Townsend at 7th                     |65          |521   |Subscriber     |94107   |
|913404 |273     |8/31/2015 20:39|San Francisco Caltrain 2 (330 Townsend)|69            |8/31/2015 20:44|Townsend at 7th                     |65          |287   |Subscriber     |94107   |
|913382 |645     |8/31/2015 20:20|Town

##  la recherche de motifs : graphe sans arcs dupliqu

In [12]:
uniqueEdges = stationGraph.edges.select("src", "dst").distinct().cache()
motifGraph = GraphFrame(stationGraph.vertices, uniqueEdges)

print("Arcs distincts : " + str(uniqueEdges.count()))

Arcs distincts : 85


## Question 11 : Tous les chemins qui forment un motif en triangle entre trois stations

In [13]:
triangles = motifGraph.find("(a)-[ab]->(b); (b)-[bc]->(c); (c)-[ca]->(a)") \
    .filter("a.id != b.id AND b.id != c.id AND a.id != c.id") \
    .cache()

print("Nombre de triangles : " + str(triangles.count()))
triangles.select(col("a.id").alias("a"), col("b.id").alias("b"), col("c.id").alias("c")).show(10, False)

Nombre de triangles : 27
+---------------------------------------+---------------------------------------+---------------------------------------+
|a                                      |b                                      |c                                      |
+---------------------------------------+---------------------------------------+---------------------------------------+
|Spear at Folsom                        |San Francisco Caltrain 2 (330 Townsend)|Townsend at 7th                        |
|Spear at Folsom                        |San Francisco Caltrain 2 (330 Townsend)|2nd at Townsend                        |
|Spear at Folsom                        |San Francisco Caltrain 2 (330 Townsend)|2nd at South Park                      |
|2nd at South Park                      |Spear at Folsom                        |San Francisco Caltrain 2 (330 Townsend)|
|2nd at South Park                      |Howard at 2nd                          |San Francisco Caltrain 2 (330 Townsend)|

## Question 12 : Tous les chemins passant par trois sommets et commençant à 'Townsend at 7th'

In [14]:
paths = motifGraph.find("(a)-[ab]->(b); (b)-[bc]->(c)") \
    .filter("a.id = 'Townsend at 7th' AND b.id != a.id AND c.id != b.id AND c.id != a.id") \
    .cache()

print("Nombre de chemins : " + str(paths.count()))
paths.select(col("a.id").alias("a"), col("b.id").alias("b"), col("c.id").alias("c")).show(10, False)

Nombre de chemins : 6
+---------------+------------------------------------+----------------------------------------+
|a              |b                                   |c                                       |
+---------------+------------------------------------+----------------------------------------+
|Townsend at 7th|Spear at Folsom                     |2nd at Townsend                         |
|Townsend at 7th|Spear at Folsom                     |San Francisco Caltrain 2 (330 Townsend) |
|Townsend at 7th|Spear at Folsom                     |Davis at Jackson                        |
|Townsend at 7th|Harry Bridges Plaza (Ferry Building)|Embarcadero at Bryant                   |
|Townsend at 7th|Harry Bridges Plaza (Ferry Building)|2nd at Townsend                         |
|Townsend at 7th|Harry Bridges Plaza (Ferry Building)|San Francisco Caltrain (Townsend at 4th)|
+---------------+------------------------------------+----------------------------------------+

